In [19]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
from natsort import natsorted
from sklearn.metrics import confusion_matrix

In [20]:
def load_file_name(dir, dataset, win_num_df):
    file_name = natsorted(glob.glob(os.path.join(dir, f"{dataset}_file_name_*.csv")))
    
    NAMES = []
    for i, id_dir in enumerate(file_name):
        # if i > 1:
        #     break
        file_df = pd.read_csv(id_dir)
        NAMES.append(file_df['file_name'].to_numpy())
        # NAMES += file_df['file_name'].to_numpy().tolist()
    window_name = []
    all_file_name = win_num_df['file_name']
    for i, name in enumerate(NAMES):
        mask = all_file_name.isin(name)
        win_num = win_num_df[mask][['file_name','window_per_person']].reset_index(drop=True)
        win_count = win_num['window_per_person'].to_numpy()
        for j, wn in enumerate(win_count):
            win_num = int(wn)
            window_name += [name[j]] * win_num

    return pd.DataFrame(window_name, columns=['file_name'])

In [21]:
def major_voted_preprocess(dir, dataset, all_person_win_num=None):
    
    label_data_dir = natsorted(glob.glob(dir + f"y_{dataset}_label*.csv"))
    proba_data_dir = natsorted(glob.glob(dir + f"yhat_{dataset}*.csv"))

    label_data_list = []
    proba_data_list = []
    for i, (label_dir, proba_dir) in tqdm(enumerate(zip(label_data_dir, proba_data_dir)), total=len(label_data_dir)):
        label_data = pd.read_csv(label_dir)
        label_data_list.append(label_data)

        proba_data = pd.read_csv(proba_dir)
        proba_data_list.append(proba_data)
    
    label_data_combine = pd.concat(label_data_list, axis=0).reset_index(drop=True)
    proba_data_combine = pd.concat(proba_data_list, axis=0).reset_index(drop=True)

    if 'file_name' not in label_data_combine.columns and 'file_name' not in proba_data_combine.columns:
        file_name = load_file_name(dir, dataset, all_person_win_num)
        label_data_combine = pd.concat([file_name, label_data_combine], axis=1)
        proba_data_combine = pd.concat([file_name, proba_data_combine], axis=1)

    label_data_combine = label_data_combine.sort_values(by=['file_name']).reset_index(drop=True)
    proba_data_combine = proba_data_combine.sort_values(by=['file_name']).reset_index(drop=True)

    sample_id, sample_counts = np.unique(label_data_combine['file_name'].to_numpy(), return_counts=True)
    sample_counts_df = pd.DataFrame({'file_name': sample_id, 'window_per_person': sample_counts})

    return label_data_combine, proba_data_combine, sample_counts_df

In [22]:
pre_combine_dir = r'D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/'
per_person_win_num_dir = r'D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result/'
save_dir=r'D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result\combine\多數決前處理/'

if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"已建立新資料夾: {save_dir}")
else:
    print(f"資料夾已存在: {save_dir}")

資料夾已存在: D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result\combine\多數決前處理/


In [23]:
h_win_num = pd.read_csv(per_person_win_num_dir + "h_window_per_person.csv")
p_win_num = pd.read_csv(per_person_win_num_dir + "p_window_per_person.csv")
all_person_win_num = pd.concat([h_win_num, p_win_num], axis=0, ignore_index=True)
all_person_win_num = all_person_win_num.sort_values(by='file_name').reset_index(drop=True)
print(all_person_win_num)

    file_name  window_per_person
0     MACE001                  9
1     MACE002                  9
2     MACE003                  9
3     MACE004                  9
4     MACE005                  9
..        ...                ...
517   MACE520                  9
518   MACE521                  9
519   MACE522                  9
520   MACE523                  9
521   MACE524                  9

[522 rows x 2 columns]


In [24]:
label_data_combine, proba_data_combine, sample_counts_df = major_voted_preprocess(pre_combine_dir, 'train', all_person_win_num=all_person_win_num)
#--------------------------------------
dataset = 'train' #train/test  記得改  記得改 記得改 記得改 記得改 記得改 記得改 記得改 記得改 記得改
#--------------------------------------
label_data_combine.to_csv(os.path.join(save_dir, f'{dataset}_label_data_combine.csv'), index=False)
proba_data_combine.to_csv(os.path.join(save_dir, f'{dataset}_proba_data_combine.csv'), index=False)
sample_counts_df.to_csv(os.path.join(save_dir, f'{dataset}_sample_counts_df.csv'), index=False)
print("全部存檔完成")

print(label_data_combine)                          #all_person_win_num 跑mutliRocket一定要開 因為檔裡面沒有file_name
print(proba_data_combine)                           #mutliRocket裡的yhat_prob沒有file_name
print(sample_counts_df)                             #skna裡的h_window_per_person.csv有file_name

100%|██████████| 90/90 [00:01<00:00, 58.87it/s]


全部存檔完成
       file_name  y_train  yhat_train  iteration
0        MACE001        0           0          2
1        MACE001        0           0         45
2        MACE001        0           0         45
3        MACE001        0           0         45
4        MACE001        0           0         45
...          ...      ...         ...        ...
418117   MACE524        0           0         22
418118   MACE524        0           0         22
418119   MACE524        0           0         22
418120   MACE524        0           0         73
418121   MACE524        0           0         85

[418122 rows x 4 columns]
       file_name  yhat_train_proba  iteration
0        MACE001          0.003312          2
1        MACE001          0.012707         45
2        MACE001          0.005309         45
3        MACE001          0.004990         45
4        MACE001          0.002885         45
...          ...               ...        ...
418117   MACE524          0.009997         22
418118   M